# Hudi Unstructured Data Quickstart — PySpark Edition

Demonstrates storing and querying unstructured data (embeddings + binary blobs) in a Hudi
table using `VECTOR` and `BLOB` column types (Hudi 1.2+).

Covers: table creation with VECTOR/BLOB columns, inserting data with synthetic embeddings,
vector similarity search, and BLOB materialization.

**Reference:** [Hudi unstructured data quickstart guide](https://hudi.apache.org/docs/unstructured-data-quick-start-guide)

## Section 1: Create a table with VECTOR and BLOB columns

In [ ]:
base_path = "/tmp/product_catalog"

# VECTOR(512) stores a 512-dim embedding. BLOB stores raw binary data.
# Hudi keeps small BLOBs inline in Parquet; larger ones as external references.
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS product_catalog (
      product_id    STRING,
      name          STRING,
      category      STRING,
      price         DECIMAL(10, 2),
      embedding     VECTOR(512),
      image         BLOB,
      created_at    TIMESTAMP,
      PRIMARY KEY (product_id) NOT ENFORCED
  ) USING hudi
  TBLPROPERTIES (
      'type'         = 'cow',
      'primaryKey'   = 'product_id'
  )
  LOCATION '{base_path}'
""")

## Section 2: Insert sample data with synthetic embeddings

In [ ]:
import random
from pyspark.sql.functions import array, lit, current_timestamp
from pyspark.sql.types import FloatType

# Synthetic 512-dim embeddings (random vectors for demo).
# In production, use a model like MobileNet or CLIP.
def random_embedding():
    return array(*[lit(random.random()).cast(FloatType()) for _ in range(512)])

products = spark.createDataFrame([
    ("P001", "Wireless Headphones", "electronics", 79.99),
    ("P002", "Running Shoes", "footwear", 129.95),
    ("P003", "Bluetooth Speaker", "electronics", 49.99),
    ("P004", "Hiking Boots", "footwear", 189.00),
    ("P005", "Laptop Stand", "accessories", 34.99),
], ["product_id", "name", "category", "price"])

products_with_embeddings = products \
    .withColumn("embedding", random_embedding()) \
    .withColumn("image", lit(bytearray(100))) \
    .withColumn("created_at", current_timestamp())

products_with_embeddings.write.format("hudi") \
    .option("hoodie.datasource.write.recordkey.field", "product_id") \
    .option("hoodie.table.name", "product_catalog") \
    .mode("Append") \
    .save(base_path)

print(f"Inserted {products_with_embeddings.count()} products with embeddings")

## Section 3: Vector similarity search

In [ ]:
# Find the 3 products most similar to a query embedding (cosine similarity).
query_vec = ", ".join([str(random.random()) for _ in range(512)])

results = spark.sql(f"""
  SELECT product_id, name, category, price
  FROM hudi_vector_search(
      'product_catalog',
      'embedding',
      ARRAY({query_vec}),
      3
  )
""")

print("Top-3 similar products:")
results.show(truncate=False)

## Section 4: Materializing BLOBs

In [ ]:
# read_blob() resolves BLOB data regardless of storage mode (inline or external).
with_blobs = spark.sql("""
  SELECT product_id, name, read_blob(image) AS image_bytes
  FROM product_catalog
  WHERE category = 'electronics'
""")

with_blobs.select("product_id", "name").show()
first_row = with_blobs.first()
print(f"Image bytes length: {len(first_row['image_bytes'])}")